# **Курсовая работа: Предиктивный анализ исходов соревновательных матчей FACEIT (CS2)**

**Автор:** Владислав Кузьменко (РГАУ-МСХА им. К.А. Тимирязева, 09.03.02 «Информационные системы и технологии»)

**Цель исследования:** Обучение и сравнительный анализ моделей классификации (Logistic Regression, Random Forest, CatBoost) для предсказания исходов поединков на соревновательной платформе FACEIT с практической апробацией на реальном профиле игрока.

## **1. Установка зависимостей и загрузка данных**

In [ ]:
# 1. Проверка и установка CatBoost (для Google Colab)
import sys, subprocess, os
try:
    import catboost
except ImportError:
    print('Установка библиотеки CatBoost...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'catboost'])
    import catboost

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

# 2. Автоматический поиск датасета (в корне Colab или в папке data)
csv_candidates = ['faceit_matches.csv', 'data/faceit_matches.csv', '../data/faceit_matches.csv', 'faceit_ml_project/data/faceit_matches.csv']
csv_path = next((p for p in csv_candidates if os.path.exists(p)), None)

if not csv_path:
    raise FileNotFoundError("Файл faceit_matches.csv не найден! Загрузите его во вкладку 'Файлы' слева.")

df = pd.read_csv(csv_path)
print(f"[УСПЕХ] Датасет загружен из: {csv_path}")
print(f"Размерность выборки: {df.shape[0]} матчей, {df.shape[1]} признаков")
df.head(5)


## **2. Разведочный анализ данных (EDA)**

In [ ]:
# Корреляционная матрица ключевых дифференциальных признаков
diff_cols = ['elo_diff', 'kd_diff', 'adr_diff', 'map_winrate_diff', 'streak_diff', 'hs_diff', 'team_1_win']
plt.figure(figsize=(9, 6), dpi=150)
sns.heatmap(df[diff_cols].corr(), annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Корреляционная матрица признаков матча FACEIT', fontsize=12, pad=12)
plt.show()

# Распределение карт пула
plt.figure(figsize=(10, 4), dpi=150)
sns.countplot(data=df, x='map_name', hue='map_name', palette='mako', legend=False, order=df['map_name'].value_counts().index)
plt.title('Распределение матчей по картам соревновательного пула CS2', fontsize=12)
plt.xlabel('Карта')
plt.ylabel('Количество сыгранных матчей')
plt.show()


## **3. Обучение и сравнительный анализ моделей машинного обучения**

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from catboost import CatBoostClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# 1. One-Hot Encoding категориального признака карт
df_enc = pd.get_dummies(df, columns=['map_name'], drop_first=True)
feature_cols = [c for c in df_enc.columns if c not in ['match_id', 'team_1_win']]
X = df_enc[feature_cols]
y = df_enc['team_1_win']

# 2. Разбиение выборки 80/20 со стратификацией
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
scaler = StandardScaler()
X_tr_sc = scaler.fit_transform(X_train)
X_te_sc = scaler.transform(X_test)

# 3. Обучение моделей
models = {
    'Logistic Regression': (LogisticRegression(max_iter=1000), X_tr_sc, X_te_sc),
    'Random Forest': (RandomForestClassifier(n_estimators=250, max_depth=12, random_state=42), X_train, X_test),
    'CatBoost': (CatBoostClassifier(iterations=400, learning_rate=0.05, depth=6, verbose=0, random_seed=42), X_train, X_test)
}

trained_models = {}
metrics_data = []

for name, (clf, xtr, xte) in models.items():
    clf.fit(xtr, y_train)
    preds = clf.predict(xte)
    probs = clf.predict_proba(xte)[:, 1]
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds)
    auc = roc_auc_score(y_test, probs)
    trained_models[name] = clf
    metrics_data.append({'Модель': name, 'Accuracy': round(acc, 4), 'F1-Score': round(f1, 4), 'ROC-AUC': round(auc, 4)})

cb_model = trained_models['CatBoost']
metrics_df = pd.DataFrame(metrics_data)
display(metrics_df)


## **4. Практическая апробация: Предиктивный анализ реального профиля игрока (-J0ker)**

Проверка обученной модели CatBoost на реальных соревновательных данных Владислава:
- **Никнейм:** `-J0ker` (в игре: `V L A D O S _ R O Y C E`)
- **Дивизион:** 4 LVL (987 ELO), 868 сыгранных матчей (415 побед, винрейт 47.81%)
- **Индивидуальные метрики:** K/D 0.93, ADR 76.53, HS% 40.0%

Модель прогнозирует вероятность победы в зависимости от выбора карты и формата лобби (соло vs дуо).

In [ ]:
# Анализ вероятности победы для профиля -J0ker
maps = ['de_anubis', 'de_ancient', 'de_dust2', 'de_mirage', 'de_inferno', 'de_nuke', 'de_vertigo']
display_maps = ['Anubis', 'Ancient', 'Dust II', 'Mirage', 'Inferno', 'Nuke', 'Vertigo']
map_wrs = {'de_anubis': 56.20, 'de_ancient': 49.50, 'de_dust2': 48.72, 'de_mirage': 47.01, 'de_inferno': 42.03, 'de_nuke': 40.96, 'de_vertigo': 25.00}

solo_preds = []
duo_preds = []

for m in maps:
    # 1. Solo Queue (случайные напарники ~990 ELO)
    row_solo = {c: 0.0 for c in feature_cols}
    team1_elo = 989.4
    team2_elo = 990.0
    team1_kd = 0.97
    team2_kd = 0.98
    team1_adr = 76.1
    team2_adr = 76.0
    team1_hs = 41.6
    team2_hs = 42.0
    team1_mwr = (map_wrs[m] + 4 * 50.0) / 5.0
    
    row_solo['team_1_avg_elo'] = team1_elo
    row_solo['team_2_avg_elo'] = team2_elo
    row_solo['elo_diff'] = team1_elo - team2_elo
    row_solo['team_1_elo_std'] = 80.0
    row_solo['team_2_elo_std'] = 80.0
    row_solo['elo_std_diff'] = 0.0
    row_solo['team_1_avg_kd'] = team1_kd
    row_solo['team_2_avg_kd'] = team2_kd
    row_solo['kd_diff'] = team1_kd - team2_kd
    row_solo['team_1_avg_adr'] = team1_adr
    row_solo['team_2_avg_adr'] = team2_adr
    row_solo['adr_diff'] = team1_adr - team2_adr
    row_solo['team_1_hs_rate'] = team1_hs
    row_solo['team_2_hs_rate'] = team2_hs
    row_solo['hs_diff'] = team1_hs - team2_hs
    row_solo['team_1_map_winrate'] = team1_mwr
    row_solo['team_2_map_winrate'] = 50.0
    row_solo['map_winrate_diff'] = team1_mwr - 50.0
    row_solo['team_1_streak'] = 0.0
    row_solo['team_2_streak'] = 0.0
    row_solo['streak_diff'] = 0.0
    row_solo['team_1_avg_matches'] = (868 + 4 * 700) / 5.0
    row_solo['team_2_avg_matches'] = 700.0
    if f'map_name_{m}' in row_solo:
        row_solo[f'map_name_{m}'] = 1.0
        
    p_solo = cb_model.predict_proba(pd.DataFrame([row_solo]))[0, 1]
    solo_preds.append(p_solo * 100)
    
    # 2. Duo Queue (Владислав + проверенный партнер KD 1.15, ELO 1050)
    row_duo = row_solo.copy()
    row_duo['team_1_avg_elo'] = 1001.4
    row_duo['elo_diff'] = 11.4
    row_duo['team_1_avg_kd'] = 1.004
    row_duo['kd_diff'] = 0.024
    row_duo['team_1_avg_adr'] = 77.8
    row_duo['adr_diff'] = 1.8
    row_duo['team_1_map_winrate'] = (map_wrs[m] + 55.0 + 3 * 50.0) / 5.0
    row_duo['map_winrate_diff'] = row_duo['team_1_map_winrate'] - 50.0
    
    p_duo = cb_model.predict_proba(pd.DataFrame([row_duo]))[0, 1]
    duo_preds.append(p_duo * 100)

res_df = pd.DataFrame({
    'Карта': display_maps,
    'Винрейт игрока (%)': [map_wrs[m] for m in maps],
    'P(Win) Solo Queue (%)': [round(x, 2) for x in solo_preds],
    'P(Win) Duo Queue (%)': [round(x, 2) for x in duo_preds]
})
display(res_df)

# Визуализация
plt.figure(figsize=(10, 4.5), dpi=150)
plt.plot(display_maps, solo_preds, marker='o', linewidth=2.5, color='#2980b9', label='Solo Queue (Соло-поиск)')
plt.plot(display_maps, duo_preds, marker='s', linewidth=2.5, color='#27ae60', linestyle='--', label='Duo Queue (с напарником K/D 1.15)')
plt.axhline(50, color='gray', linestyle=':', label='Баланс 50%')
plt.title('Предиктивный прогноз исхода матча моделью CatBoost для профиля -J0ker', fontsize=12)
plt.ylabel('Вероятность победы P(Win) (%)')
plt.legend()
plt.show()
